# Future-Guided Learning — CHB-MIT (Kaggle Edition)

Tái hiện thực nghiệm EEG của bài báo:
> Gunasekaran et al., *A predictive approach to enhance time-series forecasting*, Nature Communications 2025.
> Repo gốc: https://github.com/SkyeGunasekaran/FutureGuidedLearning

---
### ⚙️ Cấu hình Kaggle Notebook:
1. **Accelerator:** Chọn **GPU T4 x2** hoặc **GPU P100**.
2. **Internet:** Bật **Internet: ON**.


## 1. Clone repository về Kaggle Working Directory

In [ ]:
import os
import shutil

PROJECT_DIR = '/kaggle/working/FutureGuidedLearning'
REPO_URL = 'https://github.com/SkyeGunasekaran/FutureGuidedLearning.git'

if not os.path.isdir(PROJECT_DIR):
    print('Đang clone repo từ GitHub...')
    !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    print('Repo đã có trong /kaggle/working.')

%cd {PROJECT_DIR}/CHBMIT
!pwd

## 1.5. ⭐ Cải tiến tự đề xuất: thêm Bidirectional LSTM cho `CNN_LSTM_Model`

Tác giả gốc chỉ dùng LSTM 1 chiều (`nn.LSTM(..., bidirectional=False)` mặc định) trong
`CNN_LSTM_Model` — model dùng cho cả Teacher (Stage 1) và Student/FGL (Stage 3). LSTM 2 chiều
(Bidirectional) cho phép mỗi bước thời gian "thấy" được cả ngữ cảnh trước lẫn sau trong cùng
cửa sổ 30 giây, thường cải thiện phân loại chuỗi EEG — tương tự cách đổi RNN→LSTM/GRU đã cải
thiện rõ rệt trên Mackey-Glass (xem `repro/docs/EXPERIMENT_LOG.md`, mục "Lần chạy 4").

Cell dưới đây **vá trực tiếp** 3 file trong repo vừa clone (thêm tham số `bidirectional`,
mặc định `False` — không đổi hành vi gốc nếu không bật cờ `--bidirectional`), **không sửa gì
khác** (cấu trúc notebook, Stage 2 Baseline, cách nạp dữ liệu... giữ nguyên 100%).

In [ ]:
import re

model_py_path = "models/model.py"
seizure_detection_py_path = "exp/seizure_detection.py"
fgl_chbmit_py_path = "exp/FGL_CHBMIT.py"

# ---- 1) models/model.py: them tham so bidirectional vao CNN_LSTM_Model ----
with open(model_py_path, "r") as f:
    src = f.read()

old_init = """    def __init__(self, X_train_shape):
        super(CNN_LSTM_Model, self).__init__()"""
new_init = """    def __init__(self, X_train_shape, bidirectional=False):
        super(CNN_LSTM_Model, self).__init__()"""
assert old_init in src, "Khong tim thay __init__ goc cua CNN_LSTM_Model - kiem tra lai repo da clone"
src = src.replace(old_init, new_init, 1)

old_lstm = """        self.lstm = nn.LSTM(input_size=32 * 60, hidden_size=512, num_layers=3, batch_first=True)

        self.fc1 = nn.Linear(512, 256)"""
new_lstm = """        self.lstm = nn.LSTM(input_size=32 * 60, hidden_size=512, num_layers=3, batch_first=True,
                             bidirectional=bidirectional)

        self.fc1 = nn.Linear(512 * (2 if bidirectional else 1), 256)"""
assert old_lstm in src, "Khong tim thay dong khoi tao LSTM/fc1 goc - kiem tra lai repo da clone"
src = src.replace(old_lstm, new_lstm, 1)

with open(model_py_path, "w") as f:
    f.write(src)
print(f"[OK] Da va {model_py_path}: them tham so bidirectional (mac dinh False, khong doi hanh vi goc).")


def patch_exp_script(filepath, func_name, model_call_old, model_call_new,
                      argparse_anchor, call_old, call_new):
    with open(filepath, "r") as f:
        s = f.read()

    # them tham so bidirectional=False vao dinh nghia ham
    pattern = re.compile(r"(def %s\([^)]*)\)" % func_name)
    m = pattern.search(s)
    assert m, f"Khong tim thay dinh nghia ham {func_name} trong {filepath}"
    if "bidirectional" not in m.group(1):
        s = s[:m.end(1)] + ", bidirectional=False" + s[m.end(1):]

    assert model_call_old in s, f"Khong tim thay dong khoi tao model goc trong {filepath}"
    s = s.replace(model_call_old, model_call_new, 1)

    assert argparse_anchor in s, f"Khong tim thay vi tri chen --bidirectional trong {filepath}"
    new_arg = '''    parser.add_argument(
        "--bidirectional", action="store_true",
        help="Dung Bidirectional LSTM trong CNN_LSTM_Model (cai tien tu de xuat, mac dinh tat)",
    )

'''
    s = s.replace(argparse_anchor, new_arg + argparse_anchor, 1)

    assert call_old in s, f"Khong tim thay loi goi ham goc trong {filepath}"
    s = s.replace(call_old, call_new, 1)

    with open(filepath, "w") as f:
        f.write(s)
    print(f"[OK] Da va {filepath}: them co --bidirectional (mac dinh tat, khong doi hanh vi goc).")


# ---- 2) exp/seizure_detection.py (Stage 1 - Teacher) ----
patch_exp_script(
    seizure_detection_py_path,
    func_name="train_teacher_model",
    model_call_old='teacher = CNN_LSTM_Model(X_train.shape).to("cuda")',
    model_call_new='teacher = CNN_LSTM_Model(X_train.shape, bidirectional=bidirectional).to("cuda")',
    argparse_anchor="    args = parser.parse_args()",
    call_old="auc = train_teacher_model(patient, args.epochs, args.optimizer, args.patience)",
    call_new="auc = train_teacher_model(patient, args.epochs, args.optimizer, args.patience, bidirectional=args.bidirectional)",
)

# ---- 3) exp/FGL_CHBMIT.py (Stage 3 - Student/FGL) ----
patch_exp_script(
    fgl_chbmit_py_path,
    func_name="distill_student_model",
    model_call_old='student = CNN_LSTM_Model(X_train.shape).to("cuda")',
    model_call_new='student = CNN_LSTM_Model(X_train.shape, bidirectional=bidirectional).to("cuda")',
    argparse_anchor="    args = parser.parse_args()",
    call_old="""        results[patient] = distill_student_model(
            patient,
            args.epochs,
            args.trials,
            args.optimizer,
            args.alpha,
            args.temperature,
        )""",
    call_new="""        results[patient] = distill_student_model(
            patient,
            args.epochs,
            args.trials,
            args.optimizer,
            args.alpha,
            args.temperature,
            bidirectional=args.bidirectional,
        )""",
)

print("\nHoan tat va 3 file. Them --bidirectional vao lenh chay Stage 1 va Stage 3 ben duoi de dung ban cai tien.")


## 2. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q mne==1.5.1 hickle stft tqdm scikit-learn

import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

## 3. Nạp dữ liệu CHB-MIT (Chọn 1 trong 2 cách)

### 👉 CÁCH 1 (Khuyên dùng - CỰC NHANH): Dùng Dataset có sẵn trên Kaggle
1. Bấm nút **"+ Add Input"** ở góc trên bên phải notebook.
2. Tìm từ khóa **`chb-mit`** hoặc **`chb01`** và bấm dấu **`+`** để thêm dataset vào notebook.
3. Chạy cell bên dưới: code sẽ tự động quét và tạo đường dẫn tắt (symlink) trong 1 giây, **không cần tải về**!

In [ ]:
import os
import glob

# Kiểm tra các dataset đã thêm vào /kaggle/input
print('Các dataset có trong /kaggle/input:')
!ls -l /kaggle/input

# Tự động tìm thư mục chb01 hoặc các file edf trong /kaggle/input
found_chb01 = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'chb01' in dirs:
        found_chb01 = os.path.join(root, 'chb01')
        break
    # Hoặc nếu các file chb01_*.edf nằm trực tiếp trong một thư mục
    if any(f.startswith('chb01_') and f.endswith('.edf') for f in files):
        found_chb01 = root
        break

if found_chb01:
    print(f'\n[TÌM THẤY] Dữ liệu chb01 tại: {found_chb01}')
    dest_dir = 'Dataset/chb01'
    os.makedirs('Dataset', exist_ok=True)
    if os.path.islink(dest_dir) or os.path.exists(dest_dir):
        !rm -rf {dest_dir}
    os.symlink(found_chb01, dest_dir)
    print(f'-> Đã liên kết (symlink) {found_chb01} vào {dest_dir} thành công!')
    !ls -lh Dataset/chb01 | head -5
else:
    print('\n[CHƯA THẤY] Bạn chưa bấm "+ Add Input" bộ CHB-MIT nào.')
    print('Nếu không muốn thêm qua Kaggle, bạn có thể chạy Cách 2 bên dưới để tải từ PhysioNet.')

### 👉 CÁCH 2 (Dự phòng): Tải trực tiếp từ PhysioNet qua wget
*(Chỉ chạy cell này nếu bạn KHÔNG dùng Cách 1 ở trên)*

In [ ]:
# Chỉ chạy khi chưa có Dataset/chb01
if not os.path.exists('Dataset/chb01') or not any(f.endswith('.edf') for f in os.listdir('Dataset/chb01')):
    print('Đang tải dữ liệu chb01 từ PhysioNet (có thể mất vài phút)...')
    !wget -r -N -c -np -nH --cut-dirs=3 -A edf -P Dataset "https://physionet.org/files/chbmit/1.0.0/chb01/"
    !ls -lh Dataset/chb01 | head -5
else:
    print('chb01 đã sẵn sàng, không cần tải lại!')

## 4. Stage 1 — Train Teacher (Seizure Detection)

Mô hình phát hiện cơn động kinh (ictal vs interictal). Trọng số lưu tại `pytorch_models/Patient_1_detection`.

In [ ]:
!python -m exp.seizure_detection --patient 1 --epochs 50 --patience 5 --bidirectional


## 5. Stage 2 — Train Baseline (Seizure Prediction, không có FGL)

In [ ]:
!python -m exp.seizure_prediction --patient 1 --model CNN_LSTM --trials 3 --epochs 25

## 6. Stage 3 — Train Student + FGL (Seizure Prediction, CÓ Teacher dẫn đường)

In [ ]:
!python -m exp.FGL_CHBMIT --patient 1 --epochs 30 --trials 3 --optimizer Adam --alpha 0.5 --temperature 4 --bidirectional


## 7. Tổng kết kết quả & Xuất Checkpoints để tải về máy

In [ ]:
import os
import shutil

print('\n=================== KẾT QUẢ ĐỐI CHIẾU ===================')
if os.path.exists('Detection_results.txt'):
    with open('Detection_results.txt') as f:
        print('--- Teacher (Detection) Results ---')
        print(f.read().strip())

if os.path.exists('Prediction_results.txt'):
    with open('Prediction_results.txt') as f:
        print('\n--- Baseline (Prediction without FGL) Results ---')
        print(f.read().strip())

print('\n(Kết quả Student FGL được in trực tiếp ở cell Stage 3 ở trên)')
print('=========================================================')

# Nén thư mục models và các file kết quả thành file zip trong /kaggle/working
zip_output_path = '/kaggle/working/fgl_chbmit_patient01_artifacts'
if os.path.exists('pytorch_models'):
    shutil.make_archive(zip_output_path, 'zip', root_dir='.', base_dir='pytorch_models')
    print(f'\n[THÀNH CÔNG] Đã nén weights mô hình vào: {zip_output_path}.zip')
    print('👉 Bạn hãy bấm vào panel bên phải (Data > Output) để tải file zip về máy!')